# Sora Sora Obsidian Source Geochemistry
Welcome to the geochemical analysis notebook for an XRF analysis of samples from the Sora Sora Obsidian in Bolivia.  
This notebook provides the code for displaying geochemistry data together with obsidian source data from the region. It is intended to be reproducible and to provide a model for use with other data or analyses.  
Proceed through the notebook by clicking in the first cell and then press Shift-Enter to move through the notebook.  
The Python can be modified and re-run, and maps and plots are interactive and can be saved.

In [ ]:
# CELL 0 — Install required packages for local execution.
# Binder and Voila environments use environment.yml instead.
# In VS Code or plain Jupyter: run this cell once, then restart the kernel.

import sys

# Detect Binder — skip install if already in a managed environment
import os
if not os.environ.get("BINDER_LAUNCH_HOST"):
    %pip install -q \
        numpy==1.26.4 \
        pandas==2.2.1 \
        plotly==5.18.0 \
        ipywidgets==8.1.2 \
        jupyterlab_widgets==3.0.10 \
        widgetsnbextension==4.0.10 \
        anywidget==0.9.13

In [ ]:
# CELL 1 — Imports

import sys
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    from plotly.graph_objects import FigureWidget
    _plotly_figure_widget_available = True
except Exception:
    FigureWidget = go.Figure
    _plotly_figure_widget_available = False

from plotly.colors import DEFAULT_PLOTLY_COLORS
from pathlib import Path
from ipywidgets import Button, Output, VBox, SelectMultiple, Layout
import ipywidgets as widgets
from IPython.display import display, HTML

# Detect environment
def detect_environment():
    if os.environ.get("BINDER_LAUNCH_HOST"):
        return "binder"
    if os.environ.get("SERVER_SOFTWARE", "").startswith("voila"):
        return "voila"
    if os.environ.get("VSCODE_PID") or os.environ.get("TERM_PROGRAM") == "vscode":
        return "vscode"
    return "jupyter"

ENV = detect_environment()
print(f"Environment : {ENV}")
print(f"Python      : {sys.version.split()[0]}")
print(f"FigureWidget: {'available' if _plotly_figure_widget_available else 'falling back to go.Figure'}")

# Scrollable output — only inject in environments that support it
# Voila uses different CSS, so we skip or adapt
if ENV in ("jupyter", "vscode", "binder"):
    display(HTML("""
        <style>
            .output_wrapper, .output { 
                max-height: 600px !important; 
                overflow-y: auto !important; 
            }
        </style>
    """))

## Loading tables from local CSV files
### Designed to load local CSV data by default from the `../data` folder:
*  **study_samples.csv** contains geochemistry of the study samples for the current investigation.
*  **South_Am_ObsSrcs_Chem_15-25Lat_only.csv** contains obsidian source geochemistry for 15-25° latitude south. 
*  **South_Am_ObsSrcs_Locs_15-25Lat_only.csv** contains obsidian source coordinates for 15-25° latitude south. 
*  **South_Am_ObsSrcs_Locs_all.csv** contains all known obsidian source coordinates for South America. 

## Data Loading

In [ ]:
# CELL 2 - DATA LOADING
# Load geochemical data (ppm) and geological metadata
# Optional: Load tables from Google Sheets when USE_GOOGLE_SHEETS = True
USE_GOOGLE_SHEETS = False  # Toggle between Google Sheets and local CSV

# Local CSV file names in ../data
SRC_CHEM_FILE = "South_Am_ObsSrcs_Chem_15-25Lat_only.csv"
SRC_COORD_FILE = "South_Am_ObsSrcs_Locs_15-25Lat_only.csv"
STUDY_FILE = "study_samples.csv"

# SHEET_ID = unique ID from Drive URL between the two slashes after "spreadsheets/d/"
SHEET_ID = "1R4PlMACBn0l8ZguwtYDlZLnbvORzH5CHhKGFBezjSvk"

# Load data from Google Sheets or local CSV
DATA_DIR = Path("../data") # Directory for local CSV files

def read_csv_with_encodings(path):
    for encoding in ("utf-8", "latin1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    # Fallback if the file still fails with standard text encodings
    return pd.read_csv(path, encoding="latin1")

def get_df(source, sheet_name=None):
    """
    Load dataframe from Google Sheets or local CSV.
    
    Parameters
    ----------
    source : str
        Sheet ID (if USE_GOOGLE_SHEETS=True) or CSV filename
    sheet_name : str, optional
        Required when USE_GOOGLE_SHEETS=True
    """
    if USE_GOOGLE_SHEETS:
        if sheet_name is None:
            raise ValueError("sheet_name required when USE_GOOGLE_SHEETS=True")
        url = (
            f"https://docs.google.com/spreadsheets/d/{source}/gviz/tq"
            f"?tqx=out:csv&sheet={sheet_name}"
        )
        return pd.read_csv(url)
    else:
        path = DATA_DIR / source
        if not path.exists():
            raise FileNotFoundError(f"Missing local file: {path.resolve()}")
        return read_csv_with_encodings(path)

# Load datasets
if USE_GOOGLE_SHEETS:
    srcs = get_df(SHEET_ID, "KRA21_sources")
    srcs_locs = get_df(SHEET_ID, "source_coords")
    study = get_df(SHEET_ID, "samples")
else:
    srcs = get_df(SRC_CHEM_FILE)
    srcs_locs = get_df(SRC_COORD_FILE)
    study = get_df(STUDY_FILE)

# Normalize location and schema fields
rename_map = {
    "Chem_Group": "Group",
    "Latitude": "Lat",
    "Longitude": "Long",
    "Source": "Name"
}
srcs_locs = srcs_locs.rename(columns=rename_map)
if "Name" not in srcs_locs.columns and "Group" in srcs_locs.columns:
    srcs_locs["Name"] = srcs_locs["Group"].astype("string")

# Clean location table formatting and types
if "Group" in srcs_locs.columns:
    srcs_locs["Group"] = srcs_locs["Group"].astype("string").str.strip()
for col in ["Lat", "Long"]:
    if col in srcs_locs.columns:
        srcs_locs[col] = pd.to_numeric(srcs_locs[col], errors="coerce")

srcs_locs = srcs_locs.dropna(subset=[col for col in ["Lat", "Long"] if col in srcs_locs.columns]).reset_index(drop=True)

study = study.rename(columns={
    "Name": "Sample",
    "Application": "Group"
})

if "Group" not in study.columns and "Application" in study.columns:
    study = study.rename(columns={"Application": "Group"})

if "Sample" not in study.columns and "Name" in study.columns:
    study = study.rename(columns={"Name": "Sample"})

### Cleaned-up Sample Data used appears below
Proprietary data headers are stripped out (i.e., Bruker headers removed), and schema enforced.

In [21]:
# CELL 3 - DATA CLEANING
# Replace values below detection limits (often marked as <LOD or 0)
def clean_geochem_df(df):
    """Clean geochemistry dataframe headers and types."""
    # Remove Bruker artifacts and spaces
    df.columns = df.columns.str.replace(r'(Ka1|La1|\s+)', '', regex=True)
    
    # String columns
    string_cols = ['Group', 'Sample', 'Name']
    present_strings = [c for c in string_cols if c in df.columns]
    numeric_cols = df.columns.difference(present_strings)
    
    # Set dtypes
    df[present_strings] = df[present_strings].astype('string')
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    # Drop all-NaN columns
    df.dropna(axis=1, how='all', inplace=True)
    
    return df

srcs = clean_geochem_df(srcs)
study = clean_geochem_df(study)

# Debug: print dataset headers to confirm header names. 
# print('Dataset Headers')
# print('Study:', study.columns.tolist())
# print('Sources:', srcs.columns.tolist())

# Data Schema enforcement
# Define required columns for each dataset

REQUIRED = {
    "srcs": ["Sample", "Group", "Rb", "Sr", "Zr"],
    "study": ["Sample", "Group", "Rb", "Sr", "Zr"]
}
# These are minimal for this code to make sense; more elements can be present

def check_schema(df, required, name):
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

check_schema(srcs, REQUIRED["srcs"], "Sources")
check_schema(study, REQUIRED["study"], "Study")


# The study data table

print('Complete Data Types and Study Sample Table used in this visualization:')
display(study.dtypes)
display(study.head())

Complete Data Types and Study Sample Table used in this visualization:


Sample         string
Group          string
ElapsedTime     int64
Mn              int64
Rb              int64
Sr              int64
Y               int64
Zr              int64
Nb              int64
Ba              int64
Th              int64
dtype: object

,Sample,Group,ElapsedTime,Mn,Rb,Sr,Y,Zr,Nb,Ba,Th
0,RGM,Standard,30,710,151,116,29,239,23,1638,25
1,16-19215,Hearst_Bolivia,30,982,218,163,41,233,39,1496,35
2,16-19216,Hearst_Bolivia,30,1083,267,182,45,253,39,1430,36
3,16-19217,Hearst_Bolivia,30,1107,304,192,47,248,44,968,47
4,16-19218,Hearst_Bolivia,30,1022,256,170,44,260,39,1052,38


### Joined South American obsidian source chemistry with coordinates
The table below shows the source chemistry data joined to the source location coordinates by `Group`.

In [22]:
# CELL 4 - JOIN SOURCE CHEMISTRY AND LOCATIONS
# Build a joined table from source chemistry and source locations
location_columns = ["Group", "Lat", "Long", "Name"]
missing_srcs_locs = [c for c in location_columns if c not in srcs_locs.columns]
if missing_srcs_locs:
    raise KeyError(
        f"srcs_locs missing required coordinate columns: {missing_srcs_locs}. "
        f"Available columns: {srcs_locs.columns.tolist()}"
    )

srcs_locs_coords = srcs_locs[location_columns].drop_duplicates(subset=["Group"])
srcs_locs_coords = srcs_locs_coords.rename(columns={"Lat": "Lat_loc", "Long": "Long_loc"})

joined_srcs = srcs.merge(
    srcs_locs_coords,
    on="Group",
    how="left"
)

# Keep the location coordinates and drop temporary renamed columns
joined_srcs["Lat"] = joined_srcs["Lat_loc"]
joined_srcs["Long"] = joined_srcs["Long_loc"]
joined_srcs = joined_srcs.drop(columns=["Lat_loc", "Long_loc"])

print("Joined source chemistry to source locations.")
print(f"Rows: {len(joined_srcs)}")
print(f"joined_srcs columns: {joined_srcs.columns.tolist()}")

if joined_srcs[["Lat", "Long"]].isna().any(axis=None):
    print("⚠️ Some rows have missing coordinates after the join.")
else:
    print("✅ All joined rows have coordinates.")

display(joined_srcs.head(20))

Joined source chemistry to source locations.
Rows: 196
joined_srcs columns: ['Sample', 'Group', 'Mn', 'Fe', 'Zn', 'Ga', 'Th', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Peso', 'Long', 'Lat', 'Name']
✅ All joined rows have coordinates.


,Sample,Group,Mn,Fe,Zn,Ga,Th,Rb,Sr,Y,Zr,Nb,Peso,Long,Lat,Name
0,Charaña-051,CHR-BOL,310.8,6053.4,42.7,20.0,11.6,143.5,110.3,8.5,90.7,11.5,NaN,-69.16670,-17.583000,Charaña
1,Charaña-101,CHR-BOL,502.9,5802.9,48.0,14.3,10.6,129.4,106.7,8.9,86.8,10.7,NaN,-69.16670,-17.583000,Charaña
2,Charaña-102,CHR-BOL,544.6,5966.0,41.6,18.5,12.2,139.7,110.5,9.6,94.7,10.5,NaN,-69.16670,-17.583000,Charaña
3,Charaña-103,CHR-BOL,547.9,5484.7,53.9,18.3,14.5,157.6,107.3,8.9,90.3,14.7,NaN,-69.16670,-17.583000,Charaña
4,Charaña-161,CHR-BOL,484.3,6138.5,38.4,19.4,15.3,146.2,112.1,8.5,102.7,13.5,NaN,-69.16670,-17.583000,Charaña
5,Charaña-162,CHR-BOL,429.2,6031.3,39.4,14.0,13.9,138.7,109.9,7.7,97.3,12.8,NaN,-69.16670,-17.583000,Charaña
6,Charaña-163,CHR-BOL,519.7,5875.2,40.1,15.7,15.4,142.5,107.9,7.1,92.1,16.2,NaN,-69.16670,-17.583000,Charaña
7,CHIVAY-3,CHIVAY,507.5,5102.2,33.9,14.6,NaN,233.8,48.7,24.7,114.7,24.6,NaN,-71.61099,-15.640126,Chivay
8,CHIVAY-4,CHIVAY,482.8,5109.6,31.5,12.8,NaN,234.2,48.2,26.8,119.8,27.8,NaN,-71.61099,-15.640126,Chivay
9,CHIVAY-5,CHIVAY,514.9,5166.5,33.7,13.8,NaN,226.8,45.6,24.7,114.5,23.3,NaN,-71.61099,-15.640126,Chivay


### Use Lasso tool to select obsidian sources from region of interest
Click Get Selection button below to continue

In [25]:
# CELL 5 - SOURCE SELECTION WITH FIGUREWIDGET
# Monkey-patch FigureWidget delta handler to fix UID KeyError

# --- Patch function to skip deltas without 'uid' ---
def _patched_delta_handler(fig_instance):
    """
    Monkey-patch FigureWidget._handler_js2py_traceDeltas to skip deltas missing 'uid'
    This allows lasso selection to work without crashing on missing UIDs.
    """
    original_handler = fig_instance._handler_js2py_traceDeltas
    
    def safe_handler(change):
        try:
            # If trace_deltas is a list, filter to only include deltas with 'uid'
            if isinstance(change.get('new'), list):
                trace_deltas = change['new']
                deltas_with_uid = [d for d in trace_deltas if isinstance(d, dict) and 'uid' in d]
                if deltas_with_uid:
                    change_copy = dict(change)
                    change_copy['new'] = deltas_with_uid
                    try:
                        original_handler(change_copy)
                    except (KeyError, IndexError):
                        pass  # Silently skip if still fails
                # If no deltas have uid, just skip
            else:
                original_handler(change)
        except Exception:
            pass  # Silently ignore any delta processing errors
    
    fig_instance._handler_js2py_traceDeltas = safe_handler
    # Re-observe with patched handler
    if hasattr(fig_instance, '_traceDeltas'):
        fig_instance._traceDeltas.unobserve(original_handler, names='value')
        fig_instance._traceDeltas.observe(safe_handler, names='value')

# --- Data checks ---
if 'srcs_locs' not in globals():
    raise ValueError("Data not loaded: run the Data Loading cell first.")

srcs_locs = srcs_locs.copy()

if 'Lat' not in srcs_locs.columns or 'Long' not in srcs_locs.columns:
    raise ValueError("srcs_locs is missing Lat/Long columns.")

srcs_locs['Lat']  = pd.to_numeric(srcs_locs['Lat'],  errors='coerce')
srcs_locs['Long'] = pd.to_numeric(srcs_locs['Long'], errors='coerce')
srcs_locs = srcs_locs.dropna(subset=['Lat', 'Long'])

if srcs_locs.empty:
    raise ValueError("srcs_locs contains no valid coordinates after cleaning.")

names  = srcs_locs['Name'].astype(str).tolist()
groups = srcs_locs['Group'].astype(str).tolist()
lats   = srcs_locs['Lat'].astype(float).tolist()
lons   = srcs_locs['Long'].astype(float).tolist()

# DEBUG: Print first few coordinates
print(f"📍 First 3 coordinates:")
for i in range(min(3, len(names))):
    print(f"   {names[i]}: Lat={lats[i]}, Long={lons[i]}")
print(f"   Lat range: {min(lats):.2f} to {max(lats):.2f}")
print(f"   Long range: {min(lons):.2f} to {max(lons):.2f}")
print()

# --- Zoom helper ---
center_lat = np.mean(lats)
center_lon = np.mean(lons)
max_span   = max(max(lats) - min(lats), max(lons) - min(lons))
zoom = (5  if max_span > 10 else
        6  if max_span > 5  else
        7  if max_span > 2  else
        8  if max_span > 1  else
        9  if max_span > 0.5 else 11)

# --- Build trace with explicit UID ---
trace = go.Scattermapbox(
    lat=lats,
    lon=lons,
    mode='markers+text',
    text=names,
    textposition='top center',
    customdata=groups,
    marker=dict(size=10, color='steelblue'),
    selected=dict(marker=dict(size=14, color='red')),
    unselected=dict(marker=dict(opacity=0.4)),
    hovertemplate='<b>%{text}</b><br><b>Group:</b> %{customdata}<extra></extra>'
)

trace.uid = "scattermapbox_main"

# --- Create FigureWidget and apply monkey-patch ---
fig = FigureWidget(data=[trace])
_patched_delta_handler(fig)  # Apply patch to handle missing UIDs

fig.update_layout(
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=center_lat, lon=center_lon),
        zoom=zoom
    ),
    dragmode='lasso',
    height=500,
    margin=dict(r=0, l=0, t=30, b=0),
    title="🗺️ Obsidian Source Locations — lasso to select, then click Get Selection",
    hovermode='closest'
)

display(fig)

# --- Selection widget ---
output = widgets.Output()
button = widgets.Button(
    description='Get Selection',
    button_style='primary',
    icon='check',
)

selected_names = []
selected_groups = []

def on_get_selection(b):
    global selected_names, selected_groups
    selected_names = []
    selected_groups = []
    
    output.clear_output()
    
    with output:
        try:
            sel = fig.data[0].selectedpoints if hasattr(fig, 'data') and len(fig.data) > 0 else None
            
            if not sel or len(sel) == 0:
                print("⚠️  No points selected. Draw a lasso on the map first.")
                return
            
            selected_names = [names[i] for i in sel]
            selected_groups = [groups[i] for i in sel]
            
            # Remove duplicates while preserving order
            seen = set()
            selected_groups = [g for g in selected_groups if not (g in seen or seen.add(g))]
            
            print(f"✅  {len(set(selected_groups))} source(s) selected:\n")
            for g in set(selected_groups):
                print(f"  • {g}")
        except Exception as e:
            print(f"❌ Error reading selection: {e}")

button.on_click(on_get_selection)

# Fallback checkboxes
unique_groups = sorted(list(set(groups)))
checkboxes = [widgets.Checkbox(value=False, description=g) for g in unique_groups]

def on_checkbox_change(change):
    global selected_names, selected_groups
    if change['new']:
        selected = [unique_groups[i] for i, cb in enumerate(checkboxes) if cb.value]
        if selected:
            selected_names = [n for n, g in zip(names, groups) if g in selected]
            selected_groups = list(set([g for g in groups if g in selected]))

for cb in checkboxes:
    cb.observe(on_checkbox_change, names='value')

checkbox_widget = widgets.VBox(checkboxes, layout=widgets.Layout(
    border='1px solid #ccc',
    padding='10px',
    height='150px',
    overflow_y='auto'
))

print("\n📌 Fallback: Check boxes below if lasso selection isn't working:")
display(checkbox_widget)
display(button, output)

📍 First 3 coordinates:
   Caldera Vilama: Lat=-22.5667, Long=-66.6
   Cerro Kaskio: Lat=-21.382, Long=-67.7793
   Laguna Blanca Zapaleri: Lat=-22.8862, Long=-67.4359
   Lat range: -24.77 to -15.64
   Long range: -71.61 to -65.22



C:\Users\ntrip\AppData\Local\Temp\ipykernel_3956\1414611351.py:77: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



FigureWidget({
    'data': [{'customdata': [CVA2-ARG, CK-BOL, LBZ, CHR-BOL, CHIVAY, PAR-CHL, SS-
                             BOL, RAM-ARG, QRN-ARG, TOC-ARG, SALTARA, QP-CHL],
              'hovertemplate': '<b>%{text}</b><br><b>Group:</b> %{customdata}<extra></extra>',
              'lat': [-22.5667, -21.382, -22.8862, -17.583, -15.64012555,
                      -18.14205258, -18.1942, -24.77, -24.595, -23.92, -23.0833,
                      -22.647],
              'lon': [-66.6, -67.7793, -67.4359, -69.1667, -71.61098985,
                      -69.31005808, -68.2459, -65.22, -66.075, -66.08, -67.25,
                      -68.065],
              'marker': {'color': 'steelblue', 'size': 10},
              'mode': 'markers+text',
              'selected': {'marker': {'color': 'red', 'size': 14}},
              'text': [Caldera Vilama, Cerro Kaskio, Laguna Blanca Zapaleri,
                       Charaña, Chivay, Parinacota, Sora Sora, Ramadas, Quiron/La
                       Pava, Toco


📌 Fallback: Check boxes below if lasso selection isn't working:


Button(button_style='primary', description='Get Selection', icon='check', style=ButtonStyle())

Output()

ValueError: 
Invalid property path 'mapbox._derived' for layout


## Select Obsidian Sources before proceeding
Use the map above this text to select obsidian sources in your region of interest, Click "Get Selection", then click in the cell below and continue running the notebook.

In [26]:
# Display Selection results with error handling
if 'selected_names' in dir() and selected_names and len(selected_names) > 0:
    try:
        display(HTML(f"""
        <div style="padding: 10px; background-color: #c8e6c9; border-radius: 5px; border-left: 4px solid #4CAF50;">
            <b>✅ Success!</b> Selected <b>{len(selected_names)}</b> source(s)
        </div>
        """))
        
        # Create summary table with error handling
        summary = []
        for name, group in zip(selected_names, selected_groups if 'selected_groups' in dir() else [None]*len(selected_names)):
            if group in srcs['Group'].values:
                count = len(srcs[srcs['Group'] == group])
                summary.append({"Source": name, "Group": group, "Count": count})
                print(f"  {name} | {group}: {count} samples")
            else:
                print(f"  ⚠️ Warning: {name} ({group}) not found in srcs data")
        
        if len(summary) > 0:
            summary_df = pd.DataFrame(summary).sort_values("Count", ascending=False)
            display(summary_df)
        else:
            display(HTML("""
            <div style="padding: 10px; background-color: #ffcdd2; border-radius: 5px; border-left: 4px solid #f44336;">
                <b>❌ Error:</b> No selected sources found in data
            </div>
            """))
            
    except Exception as e:
        display(HTML(f"""
        <div style="padding: 10px; background-color: #ffcdd2; border-radius: 5px; border-left: 4px solid #f44336;">
            <b>❌ Error processing selection:</b> {str(e)}
        </div>
        """))
        print(f"Full error: {e}")
else:
    display(HTML("""
    <div style="padding: 10px; background-color: #fff3e0; border-radius: 5px; border-left: 4px solid #FF9800;">
        <b>⚠️ No selections yet.</b> Click the button after selecting sources on the map.
    </div>
    """))

  Cerro Kaskio | CK-BOL: 36 samples
  Laguna Blanca Zapaleri | LBZ: 41 samples
  ⚠️ Warning: Salar de Tara (SALTARA) not found in srcs data
  Quebrada Pelún | QP-CHL: 11 samples


,Source,Group,Count
1,Laguna Blanca Zapaleri,LBZ,41
0,Cerro Kaskio,CK-BOL,36
2,Quebrada Pelún,QP-CHL,11


In [27]:
# Apply selection with error handling
try:
    if 'selected_groups' not in dir() or not selected_groups:
        raise ValueError("No sources selected. Please select sources from the map first.")
    
    # Create subset using the Group values selected on the map
    srcs_subset = srcs[srcs['Group'].isin(selected_groups)].copy()

    # Prepare a normalized locations table from srcs_locs (handle varied column names)
    def _find_col(df, names):
        names_low = [n.lower() for n in names]
        for c in df.columns:
            if c.lower() in names_low:
                return c
        return None

    group_col = _find_col(srcs_locs, ['Group', 'group', 'Source', 'source']) or 'Group'
    lat_col = _find_col(srcs_locs, ['Lat', 'Latitude', 'lat', 'latitude', 'Lat_loc'])
    long_col = _find_col(srcs_locs, ['Long', 'Longitude', 'Long_loc', 'Lon', 'lon', 'longitude'])
    name_col = _find_col(srcs_locs, ['Name', 'name', 'Source', 'source'])

    if lat_col is None or long_col is None:
        raise KeyError(f"srcs_locs missing coordinate columns. Available columns: {list(srcs_locs.columns)}")

    loc_rename = {group_col: 'Group', lat_col: 'Lat', long_col: 'Long'}
    if name_col:
        loc_rename[name_col] = 'Name'
    srcs_locs_norm = srcs_locs.rename(columns=loc_rename)

    # Ensure Group exists in normalized locations
    if 'Group' not in srcs_locs_norm.columns:
        raise KeyError(f"Could not locate a 'Group' column in srcs_locs. Available: {list(srcs_locs.columns)}")

    srcs_locs_coords = srcs_locs_norm[['Group', 'Lat', 'Long'] + (['Name'] if 'Name' in srcs_locs_norm.columns else [])]
    srcs_locs_coords = srcs_locs_coords.drop_duplicates(subset=['Group']).reset_index(drop=True)

    # Now merge
    srcs_subset = srcs_subset.merge(
        srcs_locs_coords,
        on='Group',
        how='left'
    )
    
    if len(srcs_subset) == 0:
        raise ValueError(f"No data found for selected sources: {selected_groups}")
    
    if 'Lat' not in srcs_subset.columns or 'Long' not in srcs_subset.columns:
        print(f"⚠️ After merge, location columns missing. Available columns: {list(srcs_subset.columns)}")
    elif srcs_subset[['Lat', 'Long']].isna().any().any():
        print("⚠️ Some selected sources have no matching location data.")
    
    print(f"✅ Filtered to {len(srcs_subset)} samples from {len(selected_groups)} selected sources")
    
    # Count and split
    counts = srcs_subset['Group'].value_counts()
    onesample = srcs_subset[srcs_subset['Group'].map(counts) < 2]
    srcs_subset = srcs_subset[srcs_subset['Group'].map(counts) >= 2]
    
    print(f"✅ {len(srcs_subset)} samples for ellipses")
    print(f"✅ {len(onesample)} samples for points (< 2 per source)")
    
except ValueError as e:
    print(f"❌ Error: {e}")
    srcs_subset = pd.DataFrame()  # Empty dataframe
except KeyError as e:
    print(f"❌ Key error: {e}")
    srcs_subset = pd.DataFrame()
except Exception as e:
    print(f"❌ Unexpected error: {type(e).__name__}: {e}")
    srcs_subset = pd.DataFrame()

⚠️ After merge, location columns missing. Available columns: ['Sample', 'Group', 'Mn', 'Fe', 'Zn', 'Ga', 'Th', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Peso', 'Long_x', 'Lat_x', 'Lat_y', 'Long_y', 'Name']
✅ Filtered to 88 samples from 4 selected sources
✅ 88 samples for ellipses
✅ 0 samples for points (< 2 per source)


In [28]:
# This cell contains code for creating 1 s.d. confidence ellipses on biplots

def confidence_ellipse(x, y, n_std=1.96, size=100):   # Ellipses in Plotly
    """
    Get the covariance confidence ellipse of *x* and *y*.
    from https://gist.github.com/dpfoose/38ca2f5aee2aea175ecc6e599ca6e973

    Parameters
    ----------
    x, y : array-like, shape (n, )
        Input data.
    n_std : float
        The number of standard deviations to determine the ellipse's radiuses.
    size : int
        Number of points defining the ellipse
    Returns
    -------
    String containing an SVG path for the ellipse

    References (H/T)
    ----------------
    https://matplotlib.org/3.1.1/gallery/statistics/confidence_ellipse.html
    https://community.plotly.com/t/arc-shape-with-path/7205/5
    """
    if x.size != y.size:
        raise ValueError("x and y must be the same size")

    cov = np.cov(x, y)
    pearson = cov[0, 1]/np.sqrt(cov[0, 0] * cov[1, 1])
    # Using a special case to obtain the eigenvalues of this
    # two-dimensionl dataset.
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    theta = np.linspace(0, 2 * np.pi, size)
    ellipse_coords = np.column_stack([ell_radius_x * np.cos(theta), ell_radius_y * np.sin(theta)])

    # Calculating the stdandard deviation of x from
    # the squareroot of the variance and multiplying
    # with the given number of standard deviations.
    x_scale = np.sqrt(cov[0, 0]) * n_std
    x_mean = np.mean(x)

    # calculating the stdandard deviation of y ...
    y_scale = np.sqrt(cov[1, 1]) * n_std
    y_mean = np.mean(y)

    translation_matrix = np.tile([x_mean, y_mean], (ellipse_coords.shape[0], 1))
    rotation_matrix = np.array([[np.cos(np.pi / 4), np.sin(np.pi / 4)],
                                [-np.sin(np.pi / 4), np.cos(np.pi / 4)]])
    scale_matrix = np.array([[x_scale, 0],
                            [0, y_scale]])
    ellipse_coords = ellipse_coords.dot(rotation_matrix).dot(scale_matrix) + translation_matrix

    path = f'M {ellipse_coords[0, 0]}, {ellipse_coords[0, 1]}'
    for k in range(1, len(ellipse_coords)):
        path += f'L{ellipse_coords[k, 0]}, {ellipse_coords[k, 1]}'
    path += ' Z'
    return path



In [29]:
# Assign a color to each Source for consistency
name_to_color = {}

unique_groups = srcs['Group'].dropna().unique()  # drop NA just in case
colors = DEFAULT_PLOTLY_COLORS

# Cycle colors if more groups than colors
name_to_color = {name: colors[i % len(colors)] for i, name in enumerate(unique_groups)}

# 2️⃣ If you want a simple mapping to original name (optional)
unique_name_to_name = {name: name for name in unique_groups}

In [30]:
# BIPLOT
# 
# To change the variables modify x_col and y_col here
# and Re-run the cell
print('Available elements to plot.')
print(srcs_subset.columns)
print('To change elements modify x_col and y_col variables and re-run this cell:')

# Variables to plot
x_col = 'Sr'
y_col = 'Rb'
group = 'Group'

srcs_subset_valid = pd.DataFrame()
if isinstance(srcs_subset, pd.DataFrame) and not srcs_subset.empty and group in srcs_subset.columns:
    srcs_subset_valid = srcs_subset.copy()
else:
    print("⚠️ srcs_subset has no Group column or is empty. Skipping source ellipses.")

figsrcs = go.Figure()

# Helper: compute ellipse points from x,y
def _ellipse_points(x, y, n_points=120, n_std=1.0):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = (~np.isnan(x)) & (~np.isnan(y))
    x = x[mask]
    y = y[mask]
    if x.size < 2:
        return None
    cov = np.cov(x, y)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    theta = np.linspace(0, 2 * np.pi, n_points)
    unit = np.column_stack([np.cos(theta), np.sin(theta)])
    scale = n_std * np.sqrt(eigvals)
    ellipse = unit @ np.diag(scale) @ eigvecs.T
    ellipse += np.array([np.mean(x), np.mean(y)])
    return ellipse[:,0], ellipse[:,1]

# Add source ellipses as filled polygon traces (legend-friendly)
if not srcs_subset_valid.empty:
    for g in srcs_subset_valid[group].dropna().unique():
        try:
            data_g = srcs_subset_valid[srcs_subset_valid[group] == g]
            if x_col not in data_g.columns or y_col not in data_g.columns:
                print(f"⚠️ Missing {x_col}/{y_col} for group {g}; skipping")
                continue

            xvals = pd.to_numeric(data_g[x_col], errors='coerce').values
            yvals = pd.to_numeric(data_g[y_col], errors='coerce').values

            pts = _ellipse_points(xvals, yvals, n_points=120, n_std=1.0)
            if pts is None:
                # Not enough data to compute ellipse; skip
                continue
            ex, ey = pts

            color = name_to_color.get(unique_name_to_name.get(g, ""), "gray")

            # Filled polygon trace for the ellipse (shows in legend)
            figsrcs.add_trace(
                go.Scatter(
                    x=ex,
                    y=ey,
                    mode='lines',
                    fill='toself',
                    fillcolor=color,
                    line=dict(color=color, width=2),
                    opacity=0.35,
                    name=f"{g} Source Ellipse",
                    hoverinfo='skip'
                )
            )

        except Exception as e:
            print(f"⚠️ Error processing group {g}: {e}")
else:
    print('⚠️ No valid source groups available for source ellipses.')

# Add single source sample point
if isinstance(onesample, pd.DataFrame) and not onesample.empty and x_col in onesample.columns and y_col in onesample.columns:
    figsrcs.add_trace(
        go.Scatter(
            x=onesample[x_col],
            y=onesample[y_col],
            name="Source Sample",
            mode='markers',
            marker=dict(symbol='x', size=6, color='black'),
            text=onesample['Sample'] if 'Sample' in onesample.columns else None,
            hovertemplate="Source: %{text}<br><extra></extra>",
            showlegend=True
        )
    )

# Map Study sample groups to colors
study_colors = [
    name_to_color.get(unique_name_to_name.get(g, ""), "gray")
    if pd.notna(g) else 'gray'
    for g in study[group]
]

# Add study samples colored by Group (labels OFF by default)
figsrcs.add_trace(
    go.Scatter(
        x=study[x_col],
        y=study[y_col],
        name="Study Samples",
        mode='markers',
        text=study['Sample'] if 'Sample' in study.columns else None,
        hovertemplate="Sample: %{text}<br><extra></extra>",
        showlegend=True,
        marker=dict(
            size=8,
            symbol='circle',
            color=study_colors
        )
    )
)

# Index of the Study Samples trace (last trace added)
study_trace_idx = len(figsrcs.data) - 1

# Set layout with centered title and toggle button beneath legend
figsrcs.update_layout(
    title={
        'text': "Connecting Study Samples with known obsidian sources",
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title=x_col,
    yaxis_title=y_col,
    height=450,
    legend=dict(
        x=1.0,
        y=1.0,
        xanchor='left',
        yanchor='top'
    ),
    updatemenus=[
        dict(
            type='buttons',
            direction='down',
            x=1.0,
            xanchor='left',
            y=0.2,          # below the legend on the right side
            yanchor='top',
            buttons=[
                dict(
                    label='Sample Labels OFF',
                    method='restyle',
                    args=[{'mode': 'markers'}, [study_trace_idx]],
                ),
                dict(
                    label='Sample Labels ON',
                    method='restyle',
                    args=[{'mode': 'markers+text', 'textposition': 'top center'}, [study_trace_idx]],
                ),
            ],
            pad={"r": 10, "t": 10},
            showactive=True,
            bgcolor='lightgray',
            bordercolor='gray',
            borderwidth=1,
            font=dict(size=11)
        )
    ]
)

figsrcs.show()

Available elements to plot.
Index(['Sample', 'Group', 'Mn', 'Fe', 'Zn', 'Ga', 'Th', 'Rb', 'Sr', 'Y', 'Zr',
       'Nb', 'Peso', 'Long_x', 'Lat_x', 'Lat_y', 'Long_y', 'Name'],
      dtype='str')
To change elements modify x_col and y_col variables and re-run this cell:


### Biplot generated for visual source identification
Investigate biplot shown above. You may change the element variables at start of previous cell and re-run cell in order to view source ellipses and study samples using different elements on X and Y axes.
Proceed to Ternary diagram below to view three element variables at once.

In [31]:
# TERNARY PLOT CODE

# --- Function to compute ellipse in 2D (fractions space) ---
def confidence_ellipse_points(x, y, n_std=1.96, size=100):
    """
    Return Nx2 array of (x,y) points for the confidence ellipse in 2D space.
    """
    if x.size != y.size:
        raise ValueError("x and y must be the same size")

    cov = np.cov(x, y)
    mean_x = np.mean(x)
    mean_y = np.mean(y)

    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    theta = np.linspace(0, 2 * np.pi, size)
    ellipse = np.stack([np.cos(theta), np.sin(theta)], axis=1)
    scale = n_std * np.sqrt(eigvals)
    ellipse = ellipse @ np.diag(scale) @ eigvecs.T
    ellipse += np.array([mean_x, mean_y])

    return ellipse  # Nx2 array

# --- Normalize compositional data ---
def normalize_composition(df, cols):
    """
    Normalize rows so that the specified columns sum to 1
    """
    comp = df[cols].values
    row_sums = comp.sum(axis=1).reshape(-1,1)
    comp_norm = comp / row_sums
    return comp_norm

# Columns
cols = ['Rb','Sr','Zr']

# Normalize srcs and study
srcs_subset_frac = normalize_composition(srcs_subset, cols)
study_frac = normalize_composition(study, cols)

# --- Ternary Plot ---
fig = go.Figure()

# --- PLOT ELLIPSES ---
if srcs_subset_valid.empty:
    print("⚠️ No valid source subset available for ternary ellipses. Skipping source ellipses.")
else:
    srcs_subset_frac = normalize_composition(srcs_subset_valid, cols)
    for g in srcs_subset_valid[group].unique():
        color = name_to_color.get(unique_name_to_name.get(g, ""), "gray")
        
        mask = srcs_subset_valid[group] == g  # Subset mask
        rb, sr, zr = srcs_subset_frac[mask].T  # Matching indices!

        ell = confidence_ellipse_points(rb, sr)

        eRb = ell[:,0]
        eSr = ell[:,1]
        eZr = 1 - eRb - eSr

        fig.add_trace(
            go.Scatterternary(
                a=eRb, b=eSr, c=eZr,
                mode='lines',
                line=dict(color=color),
                name=f"{g} Source",
                showlegend=True
            )
        )

# --- PLOT ONESAMPLE POINTS IF NON-EMPTY ---
onesample_valid = onesample.copy() if ('Group' in onesample.columns and not onesample.empty) else pd.DataFrame()
if onesample is not None and not onesample_valid.empty:
    onesample_frac = normalize_composition(onesample_valid, cols)
    for g in onesample_valid[group].unique():
        mask = onesample_valid[group] == g
        data = onesample_frac[mask]
        color = name_to_color.get(unique_name_to_name.get(g, ""), "black")

        fig.add_trace(
            go.Scatterternary(
                a=data[:,0],
                b=data[:,1],
                c=data[:,2],
                mode='markers',
                name=f"Source Sample: {g}",
                marker=dict(symbol='x', size=6, color=color),
                text=onesample[mask]['Sample'],
                hovertemplate="Source: %{text}<extra></extra>"
            )
        )

# --- PLOT STUDY SAMPLES WITH DIFFERENT COLORS PER GROUP ---
for g in study[group].unique():
    mask = study[group] == g
    data = study_frac[mask]
    color = name_to_color.get(unique_name_to_name.get(g, ""), "gray")

    fig.add_trace(
        go.Scatterternary(
            a=data[:,0],
            b=data[:,1],
            c=data[:,2],
            mode='markers',
            name=f"Study: {g}",
            marker=dict(symbol='circle', size=8, color=color),
            text=study[mask]['Sample'],
            hovertemplate="Sample: %{text}<extra></extra>"
        )
    )

# --- Layout ---
fig.update_layout(
    ternary=dict(
        sum=1,
        aaxis=dict(title='Rb'),
        baxis=dict(title='Sr'),
        caxis=dict(title='Zr')
    ),
    title=dict(
        text="Ternary Plot with Obsidian Source Ellipses and Study Samples",
        x=0.5,
        xanchor='center',
        yanchor='top',
    ),
        height=450 #Specified height in pixels, can be modified
)

fig.show()


### Ternary Plot
Examine relationship between samples and the ellipses representing known sources. Zoom into the Ternary diagram and change the variables shown in the previous cell if you wish to view other elements in the diagram.

### The analysis notebook has concluded.
In your data exploration mouse over the points in the biplot and ternary plot to view the unique ID numbers of those samples and note their relationship to ellipses for known obsidian sources.